# Indigenous Language AI Benchmark — Group 06 Gbagyi

**Language:** Gbagyi (Gbagyi-Nkwa)  
**Group:** 06  

This notebook implements the four assignment parts: data collection, normalization/tokenization, Zipf's Law analysis, and n-gram language modeling with Laplace smoothing.

## Part 1 — Data Collection

The assignment requires at least **2,500 sentences** collected directly from online sources using Python `requests`. Replace the example URLs below with the actual Gbagyi sources selected by the group.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re, json, os
from datetime import datetime, timezone

# TODO: Replace these with real Gbagyi source URLs selected by your group.
SOURCE_URLS = [
    # 'https://example.com/gbagyi-page-1',
    # 'https://example.com/gbagyi-page-2',
]

HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; NLP student corpus collection)'}

def fetch_text(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    return soup.get_text(' ', strip=True)

def split_sentences(text):
    # Basic rule-based sentence splitting; refine if a source needs special handling.
    text = re.sub(r'\s+', ' ', text).strip()
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

raw_records = []
record_no = 1

for url in SOURCE_URLS:
    try:
        text = fetch_text(url)
        sentences = split_sentences(text)
        for sentence in sentences:
            raw_records.append({
                'id': f'gbagyi_{record_no:04d}',
                'url': url,
                'date_retrieved': datetime.now(timezone.utc).date().isoformat(),
                'raw_text': sentence
            })
            record_no += 1
    except Exception as e:
        print(f'ERROR: {url} -> {e}')

print('Raw sentence count:', len(raw_records))

In [ ]:
os.makedirs('../../data/gbagyi/raw', exist_ok=True)
RAW_PATH = '../../data/gbagyi/raw/raw_data_group_06.jsonl'

with open(RAW_PATH, 'w', encoding='utf-8') as f:
    for record in raw_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Saved:', RAW_PATH)

## Part 2 — Regex, Normalization & Tokenization

The assignment requires lowercasing while preserving diacritics/tone marks, one sentence per line, single-space token separation, detached punctuation, and a custom tokenizer.

In [ ]:
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML/XML markup
    text = re.sub(r'[\\x00-\\x08\\x0b\\x0c\\x0e-\\x1f\\x7f]', ' ', text)  # controls
    text = text.lower()
    # Detach punctuation while preserving letters, combining marks, and digits.
    text = re.sub(r'([.!?,;:!?()\[\]{}"\'\-])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def custom_tokenizer(sentence):
    # No pre-trained English tokenizer is used.
    return sentence.split()

cleaned_sentences = [normalize_text(r['raw_text']) for r in raw_records]
tokenized_sentences = [custom_tokenizer(s) for s in cleaned_sentences]

print('Example:', tokenized_sentences[:3])

### Gbagyi Stop-word List

Add at least **30 verified common Gbagyi functional words** and their English translations here. Do not invent translations; verify each item from a reliable Gbagyi linguistic/community source.

In [ ]:
STOP_WORDS = {
    # 'gbagyi_word': 'English translation',
}

print('Stop-word entries:', len(STOP_WORDS))
assert len(STOP_WORDS) >= 30, 'Add at least 30 verified Gbagyi stop words.'

In [ ]:
os.makedirs('../../data/gbagyi/processed', exist_ok=True)
PROCESSED_PATH = '../../data/gbagyi/processed/cleaned_corpus_group_06.txt'

with open(PROCESSED_PATH, 'w', encoding='utf-8') as f:
    for sentence in cleaned_sentences:
        f.write(sentence + '\n')

print('Saved:', PROCESSED_PATH)
print('Processed sentences:', len(cleaned_sentences))

## Part 3 — Zipf's Law Analysis

In [ ]:
from collections import Counter
import math
import numpy as np
import matplotlib.pyplot as plt

all_tokens = [tok for sent in tokenized_sentences for tok in sent]
freq = Counter(all_tokens)
ranked = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
ranks = np.arange(1, len(ranked) + 1)
frequencies = np.array([f for _, f in ranked])

x = np.log(ranks)
y = np.log(frequencies)
slope, intercept = np.polyfit(x, y, 1)
zipf_exponent = -slope

print('Total tokens:', len(all_tokens))
print('Unique vocabulary size (V):', len(freq))
print('Estimated Zipf exponent (s):', zipf_exponent)

plt.figure(figsize=(8, 5))
plt.loglog(ranks, frequencies, marker='.', linestyle='none')
plt.xlabel('Rank (r)')
plt.ylabel('Frequency (f)')
plt.title('Gbagyi Rank-Frequency Distribution')
plt.grid(True, which='both')
plt.show()

### Zipf synthesis

Discuss how preserved diacritics, subdot vowels, tone marks, and orthographic variation can create additional distinct token types and therefore affect vocabulary size and frequency distributions. Base the final discussion on the observed corpus statistics.

## Part 4 — Unigram & Bigram Language Models with Add-1 Smoothing

In [ ]:
from collections import defaultdict

def make_unigram_counts(sentences):
    counts = Counter()
    for sent in sentences:
        counts.update(['<s>'] + sent + ['</s>'])
    return counts

def make_bigram_counts(sentences):
    counts = Counter()
    for sent in sentences:
        tokens = ['<s>'] + sent + ['</s>']
        for a, b in zip(tokens, tokens[1:]):
            counts[(a, b)] += 1
    return counts

unigram_counts = make_unigram_counts(tokenized_sentences)
bigram_counts = make_bigram_counts(tokenized_sentences)
vocab = set(all_tokens) | {'<s>', '</s>', '<UNK>'}

def bigram_probability(w_prev, w, alpha=1):
    # Add-1 smoothing: (count(prev,w)+1)/(count(prev)+V)
    numerator = bigram_counts[(w_prev, w)] + alpha
    denominator = unigram_counts[w_prev] + alpha * len(vocab)
    return numerator / denominator

print('Vocabulary size:', len(vocab))
print('Example bigram probability:', bigram_probability('<s>', '</s>'))

In [ ]:
def perplexity(test_sentences):
    log2_sum = 0.0
    N = 0
    for sent in test_sentences:
        tokens = ['<s>'] + [w if w in vocab else '<UNK>' for w in sent] + ['</s>']
        for a, b in zip(tokens, tokens[1:]):
            p = bigram_probability(a, b)
            log2_sum += math.log2(p)
            N += 1
    return 2 ** (-log2_sum / N) if N else float('inf')

# TODO: After the instructor provides the blind test file, load it here.
TEST_PATH = '../../tests/test_gbagyi_unseen.txt'
if os.path.exists(TEST_PATH):
    with open(TEST_PATH, encoding='utf-8') as f:
        test_sentences = [custom_tokenizer(normalize_text(line)) for line in f if line.strip()]
    print('Bigram perplexity:', perplexity(test_sentences))
else:
    print('Blind test file not found yet. Do not invent a perplexity score.')

## Final Checks

- [ ] At least 2,500 real Gbagyi sentences collected directly with `requests`.
- [ ] Raw JSONL saved with `id`, `url`, `date_retrieved`, and `raw_text`.
- [ ] Cleaning preserves Gbagyi diacritics/tone marks.
- [ ] Custom tokenizer used (no pre-trained English tokenizer).
- [ ] At least 30 verified Gbagyi stop words with English translations.
- [ ] Processed corpus has exactly one sentence per line.
- [ ] Zipf log-log plot and exponent reported.
- [ ] Unigram and Bigram counts implemented from scratch.
- [ ] Add-1 smoothing implemented.
- [ ] Perplexity calculated only on the instructor-provided blind test file.
